In [6]:
import os
import shutil
#
import warnings
warnings.filterwarnings("ignore")
#
from jobflow_remote import JobController
#
def check_job_states(jc, db_id):
    """
    检查作业状态，返回不同状态和对应的作业 ID。

    :param jc: JobController 实例
    :return: 不同状态和对应作业 ID 的字典
    """
    states = {}
    db = db_id
    tem_jc = jc.get_job_doc(db_id=db)
    flow_job = jc.get_flow_info_by_job_uuid(tem_jc.uuid)

    print(jc.get_job_doc(job_id=flow_job['jobs'][0]).job.function_args[0].formula)
    print(flow_job['uuid'], flow_job['state'])
    print('#')

    for job_id in flow_job['jobs']:
        tem_jcc = jc.get_job_doc(job_id)
        state_str = str(tem_jcc.state)
        if state_str not in states:
            states[state_str] = []
        states[state_str].append(job_id)

        if state_str not in ['JobState.COMPLETED', 'JobState.WAITING', 'JobState.SUBMITTED']:
            print("#")
            print(tem_jcc.db_id, tem_jcc.state, tem_jcc.job.name)
            print(tem_jcc.run_dir)
            print("#")

    return states

def update_failed_job_incar(jc, db_id, incar_updates):
    """
    更新失败作业的 INCAR 参数并重新运行任务。

    :param jc: JobController 实例
    :param db_id: 失败作业的数据库 ID
    :param incar_updates: 要更新的 INCAR 参数字典
    """
    # Step 1: 获取失败作业的 job_doc
    job_doc = jc.get_job_doc(db_id=db_id)
    print(f"原始 INCAR 设置: {job_doc.job.maker.input_set_generator.user_incar_settings}")

    # Step 2: 使用 atomate2 的 powerups 更新作业的 INCAR 参数
    updated_job = update_user_incar_settings(job_doc.job, incar_updates)
    job_doc.job = updated_job

    # Step 3: 将更新后的作业信息写回数据库
    jc._set_job_properties(job_doc.as_db_dict(), db_id=db_id)

    # Step 4: 检查更新是否成功
    updated_job_doc = jc.get_job_doc(db_id=db_id)
    print(f"更新后的 INCAR 设置: {updated_job_doc.job.maker.input_set_generator.user_incar_settings}")

def update_failed_job_poscar(jc, db_id, snew):
    """
    更新失败作业的 POSCAR 参数并重新运行任务。

    :param jc: JobController 实例
    :param db_id: 失败作业的数据库 ID
    :param snew: 新的结构
    """
    job_doc = jc.get_job_doc(db_id=db_id)
    if 'structure' in job_doc.job.function_kwargs:
        #print(f"原始 poscar 设置: {job_doc.job.function_kwargs['structure']}")
        job_doc.job.function_kwargs['structure'] = snew
        jc._set_job_properties(job_doc.as_db_dict(), db_id=db_id)
        updated_job_doc = jc.get_job_doc(db_id=db_id)
        print('结构更新成功')
    else:
        print("Function args:", job_doc.job.function_args)
        print("Function kwargs:", job_doc.job.function_kwargs.keys())
        print('需要尝试备份再重置function_args')

def update_failed_job_resources(jc, db_id, resource_updates):
    """
    更新失败作业的 resources 参数并重新运行任务。

    :param jc: JobController 实例
    :param db_id: 失败作业的数据库 ID
    """
    job_doc = jc.get_job_doc(db_id=db_id)
    old_resources = job_doc.resources
    if old_resources:
        print(f"原始 resources 设置: {old_resources}")
        job_doc.resources = resource_updates
        jc._set_job_properties(job_doc.as_db_dict(), db_id=db_id)
        updated_job_doc = jc.get_job_doc(db_id=db_id)
        print(f"更新后的 resource 设置: {updated_job_doc.resources}")
    else:
        print('没有找到原始resources')

def handle_failed_jobs(jc, failed_job_id, project_name, new_incar, snew, new_resources):
    """
    处理失败的作业。
    """
    #
    db_id=failed_job_id
    tem_jcc = jc.get_job_doc(db_id=db_id)
    try:
        if new_incar is not None:
            update_failed_job_incar(jc, str(db_id), new_incar)
        if new_resources is not None:
            update_failed_job_resources(jc, str(db_id), new_resources)
        if snew is not None:
            update_failed_job_poscar(jc, str(db_id), snew)

        # # 清空运行目录
        # run_dir = tem_jcc.run_dir
        # for item in os.listdir(run_dir):
        #     item_path = os.path.join(run_dir, item)
        #     try:
        #         if os.path.isfile(item_path):
        #             os.remove(item_path)
        #         elif os.path.isdir(item_path):
        #             shutil.rmtree(item_path)
        #     except Exception as e:
        #         print(f"删除 {item_path} 时出错: {e}")

        # 重新运行作业
        jc.rerun_job(db_id=str(db_id), force=True)
    except Exception as e:
        print(f"处理作业 {db_id} 时出错: {e}")

In [7]:
project_name = 'wf_dr_gpu'
db_id = '11973'
jc = JobController.from_project_name(project_name)
state = check_job_states(jc, db_id)

Na2 S1
f2856957-5c0e-4e7a-9261-e8b99620f0b6 FAILED
#
#
11976 JobState.FAILED generate_slab
/fs0/home/jinzongzi/jzz/z_work/wf_dr_gpu/43/e8/7c/43e87c55-d6f9-4544-98e1-e17604954add_1
#
#
11977 JobState.FAILED generate_adslabs
/fs0/home/jinzongzi/jzz/z_work/wf_dr_gpu/c7/af/56/c7af5624-ee06-4eed-bd66-c968173dbb7b_1
#


In [26]:
db_id = '11977'
job_doc = jc.get_job_doc(db_id=db_id)

In [27]:
job_doc.job.function_kwargs['surface_idx']

'(1, 1, 1)'

In [28]:
def update_surface_index(jc, db_id, new_surface_idx):
    """
    更新作业的表面指数（surface_idx）参数。

    :param jc: JobController 实例
    :param db_id: 作业的数据库 ID
    :param new_surface_idx: 新的表面指数值
    """
    job_doc = jc.get_job_doc(db_id=db_id)
    function_kwargs = job_doc.job.function_kwargs
    if 'surface_idx' in function_kwargs:
        old_surface_idx = function_kwargs['surface_idx']
        print(f"原始 surface_idx 设置: {old_surface_idx}")
        function_kwargs['surface_idx'] = new_surface_idx
        job_doc.job.function_kwargs = function_kwargs
        jc._set_job_properties(job_doc.as_db_dict(), db_id=db_id)
        updated_job_doc = jc.get_job_doc(db_id=db_id)
        updated_surface_idx = updated_job_doc.job.function_kwargs['surface_idx']
        print(f"更新后的 surface_idx 设置: {updated_surface_idx}")
    else:
        print('没有找到 surface_idx 参数')

In [29]:
update_surface_index(jc,db_id,(1,1,1))

原始 surface_idx 设置: (1, 1, 1)
更新后的 surface_idx 设置: [1, 1, 1]


In [40]:
# surface_idx_str = str((1,1,1))
# # 去除括号
# surface_idx_str = surface_idx_str.strip('()')
# # 分割字符串并转换为整数
# surface_idx = tuple(int(x) for x in surface_idx_str.split(','))
# print(surface_idx,type(surface_idx))

import yaml

config_path = os.path.join('/Users/jzz/jzz_python/z_jupyter/1_jf/z_wf_instance/2_adsorp/inputs', "config.yaml")  # 配置文件路径固定为 inputs/config.yaml

# 检查输入文件是否存在
if not os.path.exists(config_path):
    raise FileNotFoundError(f"未找到配置文件: {config_path}")

try:
    # 1. 读取 YAML 配置文件
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
except Exception as e:
    print(f"读取配置文件时出错: {e}")

# 解析配置参数
base_incar = config.get("base_incar",)
parr_incar = config.get("parr_incar", {})
maker_kwargs = config.get("maker_kwargs", {})  # 提取 Maker.make() 专属参数
maker_config = config.get("maker")
adsorption_params = maker_config.get('adsorption_maker')
#
print(type(adsorption_params['surface_idx']))
#
if adsorption_params and 'surface_idx' in adsorption_params:
    if adsorption_params['surface_idx'] != str(None):
        surface_idx = tuple(adsorption_params['surface_idx'])
        adsorption_params['surface_idx'] = surface_idx
print(type(adsorption_params['surface_idx']),adsorption_params['surface_idx'])

<class 'str'>
<class 'str'> None


In [2]:
import json
import re
from datetime import datetime
import sys
import yaml
import os
from pymatgen.analysis.diffraction.xrd import XRDCalculator

sys.path.append('/Users/jzz/jzz_python/z_jupyter/1_jf/z_wf_instance')
from jf_jzz import (
    VASPWorkflowBuilder,
    add_metadata_to_flow,
    WorkflowSubmitter,
    to_mermaid,
    BaseVaspMaker,
    VASPConfigManager,
    DoubleRelaxMaker,
    ElectrodeInsertionMaker,
    RelaxMaker,
    StaticMaker,
    AdsorptionMaker,
)

def get_max_intensity_hkl(structure):
    # 初始化 XRDCalculator
    xrd_calculator = XRDCalculator(wavelength="CuKa")
    # 计算 XRD 图谱
    xrd_pattern = xrd_calculator.get_pattern(structure)
    # 找出强度最大的峰的索引
    max_intensity_index = np.argmax(xrd_pattern.y)
    # 找出最大强度峰对应的 hkl
    max_intensity_hkl = xrd_pattern.hkls[max_intensity_index]
    hkl = max_intensity_hkl[0]['hkl']
    # 如果是六方晶系的格式（四个指数），转换为三方晶系的格式（三个指数）
    if len(hkl) == 4:
        h, k, i, l = hkl
        new_h = h
        new_k = k
        new_l = l
        hkl = (new_h, new_k, new_l)
    return hkl

def run_workflow_from_config():
    # 固定输入输出路径（当前目录下的 inputs/outputs）
    input_dir = "inputs"
    output_dir = "outputs"
    config_path = os.path.join(input_dir, "config.yaml")  # 配置文件路径固定为 inputs/config.yaml

    # 检查输入文件是否存在
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"未找到配置文件: {config_path}")

    try:
        # 1. 读取 YAML 配置文件
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
    except Exception as e:
        print(f"读取配置文件时出错: {e}")
        return

    # 解析配置参数
    base_incar = config.get("base_incar", VASPConfigManager.get_default_incar())
    parr_incar = config.get("parr_incar", {})
    maker_kwargs = config.get("maker_kwargs", {})  # 提取 Maker.make() 专属参数
    maker_config = config.get("maker")
    adsorption_params = maker_config.get('adsorption_maker')
    specified_element = maker_config.get('specified_element')
    molecule_file = maker_config.get('molecule_file')
    input_data = config.get("input_data") 
    #
    try:
        if maker_config.get('class') == DoubleRelaxMaker.__name__:
            custom_maker = DoubleRelaxMaker()
            builder = VASPWorkflowBuilder(custom_maker)
            my_flow = builder.build_workflow(
                input_data=input_data,
                base_incar=base_incar,
                parr_incar=parr_incar,
            )
        elif maker_config.get('class') == ElectrodeInsertionMaker.__name__:
            relax_maker = RelaxMaker()  # RelaxMaker 无额外参数，直接实例化
            static_maker = StaticMaker(
                **maker_config.get("static_maker", {})  # 解包 StaticMaker 专属参数
            )
            custom_maker = ElectrodeInsertionMaker(
                relax_maker=relax_maker,
                static_maker=static_maker
            )
            builder = VASPWorkflowBuilder(custom_maker)
            my_flow = builder.build_workflow(
                input_data=input_data,
                base_incar=base_incar,
                parr_incar=parr_incar,
                **maker_kwargs  # 动态传递 Maker 专属参数（如插层参数）
            )
        elif maker_config.get('class') == AdsorptionMaker.__name__:
            # 处理 surface_idx 参数
            if adsorption_params and'surface_idx' in adsorption_params:
                surface_idx_str = str(adsorption_params['surface_idx'])
                if surface_idx_str != str(None):
                    surface_idx = tuple(int(x) for x in surface_idx_str.strip('()').split(','))
                    adsorption_params['surface_idx'] = surface_idx
                    logging.info(f"转换 surface_idx 为: {surface_idx}")
                else:
                    structure = Structure.from_file(input_data)
                    new_surface_idx = get_max_intensity_hkl(structure)
                    adsorption_params['surface_idx'] = new_surface_idx
                    logging.info(f"计算得到的 surface_idx 为: {new_surface_idx}")    
            #
            custom_maker = AdsorptionMaker(**adsorption_params)
            builder = VASPWorkflowBuilder(custom_maker)
            my_flow = builder.build_workflow_ads(
                input_data_s=input_data,
                input_data_m=molecule_file,
                specified_element=specified_element,
                base_incar=base_incar,
                parr_incar=parr_incar,
            )
        else:
            print(f"不支持的 maker 配置: {maker_config}")
            return
    except Exception as e:
        print(f"构建工作流时出错: {e}")
        return

    # 3. 添加元数据（使用配置中的 flow_identifier)，提交工作流（通用逻辑）
    try:
        my_flow = add_metadata_to_flow(
            my_flow,
            {"flow_identifier": config["flow_identifier"]},
            class_filter=BaseVaspMaker
        )
        submitter = WorkflowSubmitter()
        vis_wf = to_mermaid(my_flow)
        response = submitter.submit(
            my_flow,
            project=config["project"],
            worker_n=config["worker_n"]
        )
        print(response)
    except Exception as e:
        print(f"提交工作流时出错: {e}")
        return

    # 4. 保存结果到 outputs 目录（关键修改：固定输出目录）
    try:
        os.makedirs(output_dir, exist_ok=True)  # 自动创建 outputs 目录（若不存在）
        current_time = datetime.now().strftime("%Y%m%d%H%M%S")
        json_file_path = os.path.join(output_dir, f'output_{current_time}_{str(response[0])}.json')

        with open(json_file_path, 'w') as f:
            json.dump({
                "vis_wf": vis_wf,
                "project": config["project"],
                "flow_identifier": config["flow_identifier"],
                "response": str(response)
            }, f, indent=4)

        print(f"数据已保存到 {json_file_path}")
        return json_file_path
    except Exception as e:
        print(f"保存结果时出错: {e}")
        return


# if __name__ == "__main__":
#     run_workflow_from_config()